## Silver Layer — Transformação

Limpeza e transformação dos dados brutos pra camada Silver.

In [2]:
""" Imports """
import pandas as pd
import numpy as np
import kagglehub
from kagglehub import KaggleDatasetAdapter
from pathlib import Path

In [3]:
""" Constants """
DATASET = "blastchar/telco-customer-churn"
DATASET_FILE = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
SILVER_DIR = Path("../silver")
SILVER_OUTPUT = SILVER_DIR / "telco_churn_silver.parquet"

## 1. Carregamento dos Dados Bronze

Lê o parquet gerado pelo notebook de ingestão.

In [4]:
try:
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        DATASET,
        DATASET_FILE,
    )
except Exception as exc:
    raise RuntimeError(
        "Falha ao baixar/carregar dataset do Kaggle. Verifique suas credenciais "
        "(~/.kaggle/kaggle.json ou variáveis KAGGLE_USERNAME e KAGGLE_KEY)."
    ) from exc

print(f"Bronze carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

100%|██████████| 955k/955k [00:00<00:00, 1.21MB/s]

Bronze carregado: 7,043 linhas × 21 colunas


## 2. Auditoria de Qualidade dos Dados

Checando tipos, nulos e inconsistências.

In [ ]:
print("=== Tipos de dados ===")
df.info()

print("\n=== Nulos por coluna ===")
print(df.isnull().sum())

print(f"\n=== Strings vazias em TotalCharges ===")
blank_charges = (df["TotalCharges"] == " ").sum()
print(f"  Linhas com TotalCharges vazias: {blank_charges}")
print(f"  Essas linhas têm tenure=0: {df[df['TotalCharges'] == ' ']['tenure'].unique()}")

=== Tipos de dados ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7

Problemas encontrados: `TotalCharges` veio como texto (11 linhas vazias com tenure=0), e `SeniorCitizen` já era 0/1 enquanto o resto era Yes/No.

## 3. Correção de Tipos de Dados

Convertendo `TotalCharges` pra numérico.

In [11]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].replace(" ", np.nan), errors="coerce")

print(f"TotalCharges dtype: {df['TotalCharges'].dtype}")
print(f"Nulos em TotalCharges: {df['TotalCharges'].isnull().sum()}")

TotalCharges dtype: float64
Nulos em TotalCharges: 11


## 4. Remoção de Nulos

Removendo as 11 linhas com `TotalCharges` vazio (tenure=0).

In [12]:
linhas_antes = len(df)
df = df.dropna(subset=["TotalCharges"])
linhas_depois = len(df)

print(f"Linhas antes: {linhas_antes:,}")
print(f"Linhas removidas: {linhas_antes - linhas_depois}")
print(f"Linhas após drop: {linhas_depois:,}")

Linhas antes: 7,043
Linhas removidas: 11
Linhas após drop: 7,032


## 5. Padronização das Colunas Binárias (Yes/No → 0/1)

Convertendo Yes/No e Male/Female pra 0/1.

In [ ]:
# Colunas com Yes/No
yes_no_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]
for col in yes_no_cols:
    df[col] = (df[col] == "Yes").astype(int)

# gender: Male=1, Female=0
df["gender"] = (df["gender"] == "Male").astype(int)

print("Colunas binárias padronizadas para 0/1:")
print(df[["gender"] + yes_no_cols].head())